# CNN_5 Exp 21: Random Window Sampling

## 1. Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import resample_poly
import gc
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

## 2. ตั้งค่า

In [ ]:
SEEDS = [42, 43, 44]

PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
SOURCE_DIR = {"train": DATA_DIR, "val": DATA_DIR}
RESULT_ROOT = PROJECT_DIR / "results" / "CNN_5_7ch_Exp21_RandomWindow"
RESULT_DIR = RESULT_ROOT / "shared_data"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR = Path("/content/exp21_local")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

FS_RAW, FS = 256, 100
EPOCH_LEN_RAW, EPOCH_LEN = 30 * FS_RAW, 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN

ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS = len(ALL_CHANNELS)
N_CHANNELS = len(SELECTED_CHANNELS)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
SPLITS = ["train", "val"]
EXPECTED_SUBJECTS = {"train": 79, "val": 21}
EXPECTED_TRAIN_SEQUENCES = 10513
EXPECTED_TRAIN_CLASS_COUNTS = {"W": 1794, "N1": 351, "N2": 4134, "N3": 2409, "R": 1825}

CNN_FILTERS = [8, 16, 32]
KERNEL_SIZE = 100
DENSE_UNITS = 50
EXTRA_POOL = 25
DROPOUT = 0.5

BATCH_SIZE = 80
MAX_TRAIN_EPOCHS = 60
EVALS_PER_EPOCH = 5
LEARNING_RATE = 0.001
LR_FACTOR = 0.3
LR_PATIENCE = 5
MIN_LR = 1e-6
PATIENCE = 20

## 3.1 ฟังก์ชันโหลด Validation

In [ ]:
def load_split(split):
    files = sorted((SOURCE_DIR[split] / split).glob("*.npz"))
    if not files:
        raise FileNotFoundError(f"ไม่พบไฟล์ของ {split}")

    temp_dir = RESULT_DIR / f"temp_{split}"
    temp_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    temp_files = []
    subjects_stages = {}

    for path in files:
        subject = path.stem
        subject_X, subject_rows = [], []

        with np.load(path, allow_pickle=True) as data:
            for g, stage in enumerate(data["stage"]):
                stage = str(stage)

                if stage not in STAGES or int(data["n_valid"][g]) != SEQ_LEN:
                    continue

                y = np.asarray(data["y"][g]).astype(str).ravel()
                epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
                x = np.asarray(data["X"][g], dtype=np.float32)

                if (len(y) != SEQ_LEN or not np.all(y == stage)
                    or len(epochs) != SEQ_LEN
                    or not np.all(np.diff(epochs) == 1)
                    or x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS)
                    or not np.isfinite(x).all()):
                    continue

                x = resample_poly(
                    x[:, :, CHANNEL_IDX].reshape(-1, N_CHANNELS),
                    FS, FS_RAW, axis=0
                ).astype(np.float32)

                if x.shape != (SEQ_SAMPLES, N_CHANNELS):
                    continue

                subject_X.append(x)
                subject_rows.append({
                    "subject": subject,
                    "stage": stage,
                    "label": STAGES.index(stage),
                    "epoch_start": int(epochs[0]),
                    "epoch_end": int(epochs[-1])
                })

        stages_found = {r["stage"] for r in subject_rows}

        if stages_found == set(STAGES):
            subjects_stages[subject] = stages_found

            temp_path = temp_dir / f"{subject}.npy"
            np.save(temp_path, np.stack(subject_X).astype(np.float32))
            temp_files.append((temp_path, len(subject_X)))
            rows.extend(subject_rows)

        del subject_X, subject_rows

    if not rows:
        raise ValueError(f"{split}: ไม่พบ Subject ที่มีครบ 5 ระยะ")

    total = sum(n for _, n in temp_files)
    X_path = RESULT_DIR / f"X_{split}.npy"

    X = np.lib.format.open_memmap(
        X_path, mode="w+", dtype=np.float32,
        shape=(total, SEQ_SAMPLES, N_CHANNELS)
    )

    pos = 0
    for temp_path, n in temp_files:
        arr = np.load(temp_path, mmap_mode="r")
        X[pos:pos+n] = arr
        pos += n
        del arr

    X.flush()
    del X

    for temp_path, _ in temp_files:
        temp_path.unlink()
    temp_dir.rmdir()

    meta = pd.DataFrame(rows)
    X = np.load(X_path, mmap_mode="r")

    return X, meta

## 3.2 ฟังก์ชันโหลด Train เป็นช่วง Stage

In [ ]:
def load_train_runs():
    files = sorted((DATA_DIR / "train").glob("*.npz"))
    if not files:
        raise FileNotFoundError("ไม่พบไฟล์ train")

    temp_dir = LOCAL_DIR / "temp_train"
    temp_dir.mkdir(parents=True, exist_ok=True)

    rows, runs, temp_files = [], [], []
    row_pos = 0

    for path in files:
        subject = path.stem
        groups = []

        with np.load(path, allow_pickle=True) as data:
            for g, stage in enumerate(data["stage"]):
                stage = str(stage)
                if stage not in STAGES or int(data["n_valid"][g]) != SEQ_LEN:
                    continue

                y = np.asarray(data["y"][g]).astype(str).ravel()
                epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
                x = np.asarray(data["X"][g], dtype=np.float32)

                if (len(y) != SEQ_LEN or not np.all(y == stage)
                    or len(epochs) != SEQ_LEN
                    or not np.all(np.diff(epochs) == 1)
                    or x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS)
                    or not np.isfinite(x).all()):
                    continue

                groups.append((stage, int(epochs[0]), x[:, :, CHANNEL_IDX]))

        if {g[0] for g in groups} != set(STAGES):
            continue

        groups.sort(key=lambda g: g[1])

        subj_runs, cur = [], []
        for g in groups:
            if cur and g[0] == cur[-1][0] and g[1] == cur[-1][1] + SEQ_LEN:
                cur.append(g)
            else:
                if cur:
                    subj_runs.append(cur)
                cur = [g]
        if cur:
            subj_runs.append(cur)

        subject_X = []
        for run in subj_runs:
            k = len(run)
            raw = np.concatenate([g[2] for g in run], axis=0).reshape(-1, N_CHANNELS)
            xr = resample_poly(raw, FS, FS_RAW, axis=0).astype(np.float32)
            if xr.shape != (k * SEQ_SAMPLES, N_CHANNELS):
                raise ValueError(f"{subject}: shape หลัง resample ผิด {xr.shape}")
            subject_X.append(xr)

            runs.append({"run_start": row_pos, "k": k, "label": STAGES.index(run[0][0])})
            for j, g in enumerate(run):
                rows.append({
                    "subject": subject,
                    "stage": g[0],
                    "label": STAGES.index(g[0]),
                    "epoch_start": g[1],
                    "epoch_end": g[1] + SEQ_LEN - 1,
                    "row_start": row_pos + j * SEQ_SAMPLES,
                })
            row_pos += xr.shape[0]

        arr = np.concatenate(subject_X, axis=0)
        temp_path = temp_dir / f"{subject}.npy"
        np.save(temp_path, arr)
        temp_files.append((temp_path, arr.shape[0]))
        del groups, subject_X, arr
        gc.collect()

    X_path = LOCAL_DIR / "X_train_runs.npy"
    X = np.lib.format.open_memmap(
        X_path, mode="w+", dtype=np.float32, shape=(row_pos, N_CHANNELS)
    )

    pos = 0
    for temp_path, n in temp_files:
        arr = np.load(temp_path, mmap_mode="r")
        X[pos:pos + n] = arr
        pos += n
        del arr

    X.flush()
    del X

    for temp_path, _ in temp_files:
        temp_path.unlink()
    temp_dir.rmdir()

    return np.load(X_path, mmap_mode="r"), pd.DataFrame(rows), pd.DataFrame(runs)

## 3.3 โหลดข้อมูลและตรวจสอบก่อนฝึก

In [ ]:
DATA = {}

X, meta = load_split("val")
DATA["val"] = {"X": X, "y": meta["label"].to_numpy(np.int32), "meta": meta}
print(f"VAL: {meta['subject'].nunique()} subjects, {len(meta)} sequences, X shape={X.shape}")
print(meta["stage"].value_counts().reindex(STAGES, fill_value=0))

X_runs, train_meta, runs_df = load_train_runs()
DATA["train"] = {"y": train_meta["label"].to_numpy(np.int32), "meta": train_meta}
print(
    f"TRAIN: {train_meta['subject'].nunique()} subjects, {len(train_meta)} sequences, "
    f"{len(runs_df)} runs, X_runs shape={X_runs.shape}"
)
print(train_meta["stage"].value_counts().reindex(STAGES, fill_value=0))

y_train = DATA["train"]["y"]
X_val, y_val = DATA["val"]["X"], DATA["val"]["y"]

actual_subjects = {s: DATA[s]["meta"]["subject"].nunique() for s in DATA}
print("จำนวน Subject:", actual_subjects)
assert actual_subjects == EXPECTED_SUBJECTS

train_subjects = set(DATA["train"]["meta"]["subject"])
val_subjects = set(DATA["val"]["meta"]["subject"])
assert train_subjects.isdisjoint(val_subjects), "พบ Subject ซ้ำ!"

N_TRAIN = len(y_train)
assert N_TRAIN == EXPECTED_TRAIN_SEQUENCES, f"จำนวน Train = {N_TRAIN}"

cls_counts = {s: int((train_meta["stage"] == s).sum()) for s in STAGES}
assert cls_counts == EXPECTED_TRAIN_CLASS_COUNTS, cls_counts
assert runs_df["k"].sum() == N_TRAIN

print("Train:", N_TRAIN, "| Val:", len(y_val))
print("ตรวจสอบข้อมูลผ่าน")

## 3.4 ตัวสุ่มหน้าต่าง Train

In [ ]:
RUN_START = runs_df["run_start"].to_numpy(np.int64)
RUN_K = runs_df["k"].to_numpy(np.int64)
ORIG_STARTS = train_meta["row_start"].to_numpy(np.int64)


def get_windows(starts):
    return np.stack([X_runs[s:s + SEQ_SAMPLES] for s in starts])


def sample_train_starts(rng):
    starts = np.empty(N_TRAIN, np.int64)
    p = 0
    for rs, k in zip(RUN_START, RUN_K):
        n_off = SEQ_LEN * k - SEQ_LEN + 1
        offs = rng.choice(n_off, size=k, replace=False)
        starts[p:p + k] = rs + offs * EPOCH_LEN
        p += k
    if p != N_TRAIN:
        raise RuntimeError(f"จำนวนหน้าต่าง {p} != {N_TRAIN}")
    return starts


test_starts = sample_train_starts(np.random.default_rng(0))
run_end_per_window = np.repeat(RUN_START + RUN_K * SEQ_SAMPLES, RUN_K)
run_start_per_window = np.repeat(RUN_START, RUN_K)

assert ((test_starts >= run_start_per_window) & (test_starts + SEQ_SAMPLES <= run_end_per_window)).all(), "หน้าต่างข้ามขอบเขตช่วง"
assert get_windows(test_starts[:4]).shape == (4, SEQ_SAMPLES, N_CHANNELS) == (4, 15000, 7)

print(f"รอบทดสอบ: {(test_starts != ORIG_STARTS).mean():.1%} ของหน้าต่างต่างจากตำแหน่งเดิม")
print("ตรวจสอบตัวสุ่มผ่าน")

## 3.5 ตรวจ Resample ทั้งช่วงเทียบกับ Resample ทีละ Sequence

In [ ]:
check_rng = np.random.default_rng(0)
check_subjects = check_rng.choice(sorted(train_subjects), size=10, replace=False)

CENTER_TOL = 1e-3
EDGE_MARGIN = 50

center_max, edge_max, n_checked = 0.0, 0.0, 0

for subject in check_subjects:
    sub_meta = train_meta[train_meta["subject"] == subject]
    row_of = dict(zip(sub_meta["epoch_start"], sub_meta["row_start"]))
    checked_here = 0

    with np.load(DATA_DIR / "train" / f"{subject}.npz", allow_pickle=True) as data:
        for g, stage in enumerate(data["stage"]):
            if checked_here >= 20:
                break

            epochs = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
            if str(stage) not in STAGES or int(data["n_valid"][g]) != SEQ_LEN or int(epochs[0]) not in row_of:
                continue

            x = np.asarray(data["X"][g], dtype=np.float32)
            ref = resample_poly(
                x[:, :, CHANNEL_IDX].reshape(-1, N_CHANNELS), FS, FS_RAW, axis=0
            ).astype(np.float32)

            r = row_of[int(epochs[0])]
            crop = X_runs[r:r + SEQ_SAMPLES]
            d = np.abs(ref - crop)

            center_max = max(center_max, float(d[EDGE_MARGIN:-EDGE_MARGIN].max()))
            edge_max = max(edge_max, float(max(d[:EDGE_MARGIN].max(), d[-EDGE_MARGIN:].max())))
            n_checked += 1
            checked_here += 1

print("ตรวจแล้ว", n_checked, "sequences จาก", len(check_subjects), "subjects")
print("ความต่างสูงสุดส่วนกลาง:", center_max)
print("ความต่างสูงสุดบริเวณขอบ:", edge_max)

assert n_checked > 0
assert center_max <= CENTER_TOL, "ส่วนกลางต่างเกินเกณฑ์"
print("ตรวจ resample ผ่าน")

## 3.6 Class Weight

In [ ]:
counts = np.bincount(y_train, minlength=len(STAGES))

assert np.all(counts > 0), "พบคลาสที่ไม่มีข้อมูลใน Train"

class_weights = 1.0 / np.power(counts, 0.65)
class_weights = class_weights / class_weights.mean()

print("Class counts:")
print(pd.Series(counts, index=STAGES))

print("\nClass weights (Inverse Frequency):")
print(pd.Series(class_weights, index=STAGES))

## 4. สร้างโมเดล CNN_5

In [ ]:
def build_cnn5():
    inputs = layers.Input(shape=(SEQ_SAMPLES, N_CHANNELS))
    x = layers.Permute((2, 1))(inputs)
    x = layers.Reshape((N_CHANNELS, SEQ_SAMPLES, 1))(x)

    for n_filters in CNN_FILTERS:
        x = layers.Conv2D(n_filters, (1, KERNEL_SIZE), strides=1, padding="same")(x)
        x = layers.ReLU()(x)
        x = layers.BatchNormalization()(x)
        x = layers.AveragePooling2D(pool_size=(1, 2), strides=(1, 2))(x)

    x = layers.AveragePooling2D(pool_size=(1, EXTRA_POOL), strides=(1, EXTRA_POOL))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(DENSE_UNITS)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(len(STAGES), activation="softmax")(x)

    return models.Model(inputs, outputs, name="CNN_5")

## 6. ฟังก์ชันคำนวณ Metric

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def class_analysis(cm):
    rows = []
    for i, stage in enumerate(STAGES):
        errors = cm[i].copy()
        errors[i] = 0
        j = errors.argmax()
        rows.append({
            "stage": stage,
            "correct": int(cm[i, i]),
            "total": int(cm[i].sum()),
            "recall": cm[i, i] / cm[i].sum(),
            "most_confused_with": STAGES[j] if errors[j] > 0 else "-",
            "confused_count": int(errors[j]),
        })
    return pd.DataFrame(rows)


def make_prediction_table(split, prob):
    meta = DATA[split]["meta"]
    pred = prob.argmax(axis=1)
    table = meta[["subject", "epoch_start", "epoch_end"]].copy()
    table["actual"] = [STAGES[i] for i in meta["label"]]
    table["predicted"] = [STAGES[i] for i in pred]
    table["confidence"] = prob.max(axis=1)
    for i, stage in enumerate(STAGES):
        table["prob_" + stage] = prob[:, i]
    table["correct"] = table["actual"] == table["predicted"]
    return table

In [ ]:
def get_chunk(split, i, j):
    if split == "train":
        return get_windows(ORIG_STARTS[i:j])
    return DATA["val"]["X"][i:j]

## 5, 7, 8. ฝึก ประเมิน และสรุปผลต่อ Seed

In [ ]:
def run_seed(seed):
    result_dir = RESULT_ROOT / f"seed{seed}"
    done_path = result_dir / "experiment_row.csv"
    if done_path.exists():
        print(f"seed {seed}: มีผลอยู่แล้ว ข้าม")
        return pd.read_csv(done_path)
    result_dir.mkdir(parents=True, exist_ok=True)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    win_rng = np.random.default_rng(seed + 1000)

    print(f"\n===== seed {seed} =====")

    model = build_cnn5()
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE, momentum=0.9),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    assert model.input_shape == (None, 15000, 7)
    n_train = len(y_train)
    batches_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
    eval_every = max(1, round(batches_per_epoch / EVALS_PER_EPOCH))

    evals_per_epoch = batches_per_epoch / eval_every

    print("batches per training epoch:", batches_per_epoch)
    print("validate every", eval_every, "batch(es)")
    print("validations per training epoch:", evals_per_epoch)
    print("patience:", PATIENCE, "validations =", PATIENCE / evals_per_epoch, "training epochs")

    history = []
    best_val_loss = np.inf
    best_weights = model.get_weights()
    wait = 0
    lr_wait = 0
    lr = LEARNING_RATE
    step = 0
    stop = False

    for epoch in range(MAX_TRAIN_EPOCHS):
        win_starts = sample_train_starts(win_rng)
        order = np.random.permutation(n_train)

        for b in range(batches_per_epoch):
            idx = order[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
            xb = get_windows(win_starts[idx])
            yb = y_train[idx]
            result = model.train_on_batch(xb, yb, sample_weight=class_weights[yb], return_dict=True)
            step += 1

            if step % eval_every == 0:
                val_loss, val_acc = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
                history.append({
                    "step": step,
                    "train_epoch": step / batches_per_epoch,
                    "lr": lr,
                    "train_last_batch_loss": result["loss"],
                    "train_last_batch_accuracy": result["accuracy"],
                    "val_loss": val_loss,
                    "val_accuracy": val_acc,
                })

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_weights = model.get_weights()
                    wait = 0
                    lr_wait = 0
                else:
                    wait += 1
                    lr_wait += 1

                if lr_wait >= LR_PATIENCE:
                    lr = max(lr * LR_FACTOR, MIN_LR)
                    model.optimizer.learning_rate.assign(lr)
                    lr_wait = 0

                if wait >= PATIENCE:
                    stop = True
                    break

        print(f"epoch {epoch + 1}: val_loss={history[-1]['val_loss']:.4f}, val_acc={history[-1]['val_accuracy']:.4f}, best_val_loss={best_val_loss:.4f}, lr={lr:.6f}")

        if stop:
            print("early stopping")
            break

    model.set_weights(best_weights)
    model.save(result_dir / "cnn5_best_model.keras")

    history_df = pd.DataFrame(history)
    history_df.to_csv(result_dir / "training_history.csv", index=False)

    best_step = history_df.loc[history_df["val_loss"].idxmin(), "step"]
    print("best checkpoint at step", int(best_step), "| val_loss", round(best_val_loss, 4))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, name in zip(axes, ["loss", "accuracy"]):
        ax.plot(history_df["step"], history_df[f"train_last_batch_{name}"], label="train (last batch)")
        ax.plot(history_df["step"], history_df[f"val_{name}"], label="validation")
        ax.axvline(best_step, color="gray", linestyle="--", label="best checkpoint")
        ax.set_xlabel("Training update step")
        ax.set_ylabel(name)
        ax.legend()
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(result_dir / "training_curves.png", dpi=150)
    plt.show()
    results = {}
    tables = {}

    for split in ["train", "val"]:
        y = DATA[split]["y"]
        CHUNK = 1000
        prob = np.concatenate([
            model.predict(get_chunk(split, i, min(i + CHUNK, len(y))), batch_size=BATCH_SIZE, verbose=0)
            for i in range(0, len(y), CHUNK)
        ])
        y_pred = prob.argmax(axis=1)

        overall, per_class = calc_metrics(y, y_pred)
        results[split] = overall
        tables[split] = make_prediction_table(split, prob)

        if split == "train":
            continue

        cm = confusion_matrix(y, y_pred, labels=LABELS)
        cm_norm = confusion_matrix(y, y_pred, labels=LABELS, normalize="true")

        print("=====", split.upper(), "=====")
        display(pd.DataFrame([overall]).round(4))

        print("Per-class metrics")
        display(per_class.round(4))

        print("Per-stage confusion (one-vs-rest)")
        display(stage_confusion(cm).round(4))

        print("Per-class analysis")
        display(class_analysis(cm).round(4))

        fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
        ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
        axes[0].set_title("Confusion matrix (count)")
        ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
        axes[1].set_title("Confusion matrix (normalized by actual)")
        plt.suptitle(split)
        plt.tight_layout()
        plt.savefig(result_dir / f"{split}_confusion_matrix.png", dpi=150)
        plt.show()

        x_pos = np.arange(len(STAGES))
        plt.figure(figsize=(8, 4.5))
        plt.bar(x_pos - 0.27, per_class["precision"], 0.27, label="Precision")
        plt.bar(x_pos, per_class["recall"], 0.27, label="Recall")
        plt.bar(x_pos + 0.27, per_class["f1_score"], 0.27, label="F1-score")
        plt.xticks(x_pos, STAGES)
        plt.ylim(0, 1.05)
        plt.title(f"{split}: metrics per stage")
        plt.legend()
        plt.grid(axis="y", alpha=0.4)
        plt.tight_layout()
        plt.savefig(result_dir / f"{split}_per_class_metrics.png", dpi=150)
        plt.show()

        pd.DataFrame([overall]).to_csv(result_dir / f"{split}_overall_metrics.csv", index=False)
        per_class.to_csv(result_dir / f"{split}_per_class_metrics.csv", index=False)
        stage_confusion(cm).to_csv(result_dir / f"{split}_per_stage_confusion.csv", index=False)
        class_analysis(cm).to_csv(result_dir / f"{split}_class_analysis.csv", index=False)
        pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(result_dir / f"{split}_confusion_matrix.csv")
        tables[split].to_csv(result_dir / f"{split}_predictions.csv", index=False)

    summary = pd.DataFrame(results).T
    summary.index.name = "split"
    summary.to_csv(result_dir / "summary.csv")
    display(summary.round(4))
    summary_row = pd.DataFrame([{
        "experiment": "CNN_5_7ch_Exp21",
        "seed": seed,
        "changed_factor": "random epoch",
        "n_train_sequences": len(y_train),
        "trainable_params": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        "best_val_loss": round(best_val_loss, 4),
        "val_accuracy": round(results["val"]["accuracy"], 4),
        "val_macro_f1": round(results["val"]["macro_f1"], 4),
        "val_kappa": round(results["val"]["cohen_kappa"], 4),
        "train_acc_minus_val_acc": round(results["train"]["accuracy"] - results["val"]["accuracy"], 4),
    }])
    display(summary_row)
    summary_row.to_csv(result_dir / "experiment_row.csv", index=False)
    return summary_row

## 9. รันทั้ง 3 Seeds และสรุปรวม

In [ ]:
rows = [run_seed(s) for s in SEEDS]

all_seeds = pd.concat(rows, ignore_index=True)
all_seeds.to_csv(RESULT_ROOT / "exp21_all_seeds_summary.csv", index=False)
display(all_seeds)
display(all_seeds[["best_val_loss", "val_accuracy", "val_macro_f1", "val_kappa"]].agg(["mean", "std"]).round(4))

In [ ]:

from IPython.display import display, Image


RESULT_ROOT = PROJECT_DIR / "results" / "CNN_5_7ch_Exp21_RandomWindow"
SEEDS = [42, 43, 44]


def find_seed_dir(seed):
    candidates = [
        RESULT_ROOT / f"seed{seed}",
        PROJECT_DIR / "results" / f"CNN_5_7ch_Exp21_RandomWindow_seed{seed}",
    ]
    for d in candidates:
        if (d / "experiment_row.csv").exists():
            return d
    return None


def show_saved(seed):
    d = find_seed_dir(seed)
    if d is None:
        print(f"seed {seed}: ยังไม่มีผลที่บันทึกไว้")
        return None

    print(f"===== seed {seed} ({d.name}) =====")
    display(pd.read_csv(d / "experiment_row.csv"))

    for name in ["val_overall_metrics", "val_per_class_metrics", "val_per_stage_confusion", "val_class_analysis", "summary"]:
        f = d / f"{name}.csv"
        if f.exists():
            print(name)
            display(pd.read_csv(f).round(4))

    for name in ["training_curves", "val_confusion_matrix", "val_per_class_metrics"]:
        f = d / f"{name}.png"
        if f.exists():
            display(Image(filename=str(f)))

    return pd.read_csv(d / "experiment_row.csv")


saved_rows = [show_saved(s) for s in SEEDS]
saved_rows = [r for r in saved_rows if r is not None]

if saved_rows:
    all_saved = pd.concat(saved_rows, ignore_index=True)
    display(all_saved)
    display(all_saved[["best_val_loss", "val_accuracy", "val_macro_f1", "val_kappa"]].agg(["mean", "std"]).round(4))